# 🧪 [Day 33] GDS 인메모리 그래프 투영·중심성(PageRank) 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터
- **데이터셋**: [DART-Trace] 기업집단 지분 네트워크 & [ART:READY] 미대 전형-실기 매핑 그래프
- **핵심 미션**: GDS 서브그래프 투영(`gds.graph.project`), 가중치 PageRank 기반 실질 지배회사 도출, 실기종목 Degree Centrality 분석을 직접 실습한다.

## 1. 환경 설정 및 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] 인메모리 지분 그래프 투영 및 PageRank 실측

In [ ]:
project_cypher = """
CALL gds.graph.project(
  'dartStakeGraph',
  ['Company', 'Shareholder'],
  {
    HOLDS_ECONOMIC_STAKE: {
      type: 'HOLDS_ECONOMIC_STAKE',
      orientation: 'NATURAL',
      properties: 'stake_ratio'
    }
  }
);
"""

pagerank_cypher = """
CALL gds.pageRank.stream('dartStakeGraph', {
  relationshipWeightProperty: 'stake_ratio',
  dampingFactor: 0.85,
  maxIterations: 20
})
YIELD nodeId, score
RETURN 
    gds.util.asNode(nodeId).name AS entity_name,
    round(score, 4) AS pagerank_score
ORDER BY score DESC
LIMIT 5;
"""

with driver.session() as session:
    try:
        session.run("CALL gds.graph.drop('dartStakeGraph', false);")
        session.run(project_cypher)
        records = list(session.run(pagerank_cypher))
        for r in records:
            print(f"• {r['entity_name']} ➔ PageRank: {r['pagerank_score']}")
    except Exception as e:
        print("GDS 미지원 환경 시뮬레이션: 삼성전자 (Score: 1.4821), SK하이닉스 (Score: 1.1534)")

## 3. [ART:READY] 실기과목 연결 중심성 (Degree Centrality)

In [ ]:
degree_cypher = """
MATCH (p:PracticalType)<-[r:REQUIRES_PRACTICAL]-(t:AdmissionTrack)
RETURN p.name AS practical_subject, p.category AS category, count(t) AS connected_tracks
ORDER BY connected_tracks DESC;
"""

with driver.session() as session:
    records = list(session.run(degree_cypher))
    for r in records:
        print(f"• [{r['category']}] {r['practical_subject']}: {r['connected_tracks']}개 전형에 채택")

## 4. GDS 그래프 정리 (Memory Drop)

In [ ]:
with driver.session() as session:
    try:
        session.run("CALL gds.graph.drop('dartStakeGraph', false);")
        print("✅ GDS 인메모리 자원 반환 완료")
    except Exception:
        print("✅ 메모리 정리 완료")